# pandas 01. 読み込みと型

**引数ひとつで結果が変わる**ところだけを集めてある。上から順に、
セルを実行する前に「どうなるか」を予想してから実行する。

予想と違ったところが、自分がまだ知らないところ。

- 元データは `/data/orders.csv`。12行しかないので全部目で見られる
- 各節の最後に `<details>` で解説がある。**先に開かない**

In [ ]:
import io
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

# まず生のテキストをそのまま見る。pandas を通す前の姿を知っておく
print(open("/data/orders.csv", encoding="utf-8").read())

---
## 1. 型を推測させると何が起きるか

### 1-1. 素で読む vs dtype=str

`read_csv` は既定で列の型を推測する。推測は親切に見えて、汚れを勝手に埋める。

**実行する前に、違いを予想する。**

In [ ]:
# A: 素で読む
a = pd.read_csv("/data/orders.csv")
display(a.head(8))
a.dtypes

In [ ]:
# B: 全部文字列で読む
b = pd.read_csv("/data/orders.csv", dtype=str)
display(b.head(8))
b.dtypes

<details>
<summary>何が起きたか</summary>

`A` では:

- `qty` が **float64**。欠損が1つあるだけで整数列が浮動小数になる。
  pandas の既定の `int64` は欠損を表現できないため
- `amount` が **object**(文字列)。`"2,400"` と `N/A` が混ざっているので
  数値にできず、全体が文字列のまま。**この列を `sum()` すると文字列連結になる**
- `order_date` も **object**。日付として解釈されていない

`B` では全部 `object`。これは「推測させない」という明示的な選択で、
汚れたデータを扱うときの出発点。**まず全部文字列で受けて、自分で解釈する。**

</details>

### 1-2. IDの先頭ゼロ

推測が壊すものの代表例。`read_csv` に小さな文字列を直接食わせて確かめる。

**実行する前に、違いを予想する。**

In [ ]:
# A: 推測させる
csv = "code,name\n00123,foo\n00456,bar\n"
a = pd.read_csv(io.StringIO(csv))
display(a)
a.dtypes

In [ ]:
# B: 文字列で読む
b = pd.read_csv(io.StringIO(csv), dtype={"code": str})
display(b)
b.dtypes

<details>
<summary>何が起きたか</summary>

`A` は `00123` を整数 `123` にする。**先頭のゼロが消える。**

一度消えると、元の `00123` を持っている別のテーブルと結合できなくなる。
しかもエラーは出ない。結合結果が0件になって初めて気づく。

**IDは常に文字列で扱う。** これは例外なく守ってよい規則。

</details>

---
## 2. 欠損をどう受け取るか

### 2-1. keep_default_na

pandas は既定でいくつかの文字列を勝手に欠損(NaN)にする。何が対象かを知っておく。

**実行する前に、違いを予想する。**

In [ ]:
# A: 既定 (keep_default_na=True)
a = pd.read_csv("/data/orders.csv", dtype=str)
display(a[["order_id", "customer_id", "amount", "qty"]])
a.isna().sum()

In [ ]:
# B: 何も欠損扱いしない
b = pd.read_csv("/data/orders.csv", dtype=str, keep_default_na=False)
display(b[["order_id", "customer_id", "amount", "qty"]])
b.isna().sum()

<details>
<summary>何が起きたか</summary>

`A` では `N/A` と空欄が `NaN` になっている。pandas の既定リストには
`NA` `N/A` `NULL` `NaN` `nan` `None` `-` **ではないもの**など約20種が入っている
(`-` は入っていない。`pd._libs.parsers.STR_NA_VALUES` で全部見られる)。

`B` では**何も欠損にしない**。`N/A` は文字列 `'N/A'` のまま、空欄は `''` のまま。

どちらが良いかは場合による。ただし **`A` は「pandas が知っている表現」しか拾わない**。
`-` や `不明` や `999` は素通りする。結局あとで自分で判定するなら、
最初から `B` にして、欠損の定義を1箇所に集めたほうが読みやすい。

</details>

### 2-2. na_values で足す vs 自分で判定する

欠損表現がデータ固有なとき、どこで吸収するか。

**実行する前に、違いを予想する。**

In [ ]:
# A: read_csv に教える
a = pd.read_csv("/data/orders.csv", dtype=str,
                keep_default_na=False, na_values=["N/A", "-", ""])
display(a[["customer_id", "amount"]])
a.isna().sum()

In [ ]:
# B: 読んだあとで置き換える
NULLISH = {"NULL", "N/A", "-", ""}
b = pd.read_csv("/data/orders.csv", dtype=str, keep_default_na=False)
b = b.map(lambda s: None if str(s).strip() in NULLISH else s)
display(b[["customer_id", "amount"]])
b.isna().sum()

<details>
<summary>何が起きたか</summary>

結果は同じ。違うのは**吸収する場所**。

- `A` は読み込み時に閉じる。短い。ただしファイルごとに `read_csv` の引数が散る
- `B` は「欠損の定義」がコードの中に1つある。**前後の空白も落とせる**
  (`" N/A "` は `A` では拾えない)

複数ファイルを読むなら `B` のほうが揃えやすい。
`DataFrame.map` は全セルに関数を適用する(pandas 2.1 以前は `applymap`)。

</details>

---
## 3. 数値に変える

### 3-1. astype vs to_numeric(errors=)

文字列を数値にする2つのやり方。失敗したときの挙動が違う。

**実行する前に、違いを予想する。**

In [ ]:
# A: astype
s = pd.Series(["1200", "980", "N/A", "2,400", "0"])
try:
    print(s.astype(int))
except Exception as e:
    print(f"{type(e).__name__}: {e}")

In [ ]:
# B: to_numeric(errors='coerce')
s = pd.Series(["1200", "980", "N/A", "2,400", "0"])
print(pd.to_numeric(s, errors="coerce"))

<details>
<summary>何が起きたか</summary>

- `A` は**落ちる**。最初に失敗した値を教えてくれる
- `B` は失敗した値を**黙って `NaN` にする**。落ちない代わりに、
  何件死んだかを自分で数えないと気づけない

`"2,400"` は `B` でも `NaN` になっている点に注意。
**カンマは自動では外れない。** `errors="coerce"` は「読めないものを捨てる」だけで、
「読めるように直す」わけではない。

どちらを使うかは意図で決める:

- 変換できないものは**あってはならない** → `astype`(落ちてほしい)
- 変換できないものが**あることを想定している** → `to_numeric(errors="coerce")` の後に
  `isna().sum()` で件数を必ず数える

`errors="raise"`(既定)なら `to_numeric` も落ちる。

</details>

### 3-2. thousands= で読む

`"2,400"` を数値にする方法。read_csv 側で解決することもできる。

**実行する前に、違いを予想する。**

In [ ]:
# A: thousands なし
csv = "amount\n1200\n\"2,400\"\n980\n"
a = pd.read_csv(io.StringIO(csv))
print(a.dtypes)
a

In [ ]:
# B: thousands=','
csv = "amount\n1200\n\"2,400\"\n980\n"
b = pd.read_csv(io.StringIO(csv), thousands=",")
print(b.dtypes)
b

<details>
<summary>何が起きたか</summary>

`B` は `int64` になる。`thousands=","` は**桁区切りとしてのカンマ**を外してくれる。

ただし効くのは「その列全体が数値として解釈できる」ときだけ。
`N/A` が混ざっていると `object` に戻る。実データでは
`thousands` だけで片付くことは少なく、結局は自分で正規化することになる。

**引数で済むなら引数で済ませ、済まないなら自分で書く。**
どちらでも良いが、**両方書いて二重に処理しない**こと。

</details>

---
## 4. 整数と欠損

### 4-1. int64 と Int64

欠損を含む整数列をどう持つか。大文字の `Int64` は別物。

**実行する前に、違いを予想する。**

In [ ]:
# A: 既定の型
a = pd.read_csv("/data/orders.csv")
print(a["qty"].dtype)
display(a["qty"].head(6))
print("合計:", a["qty"].sum())

In [ ]:
# B: nullable な整数型
b = pd.read_csv("/data/orders.csv")
b["qty"] = b["qty"].astype("Int64")
print(b["qty"].dtype)
display(b["qty"].head(6))
print("合計:", b["qty"].sum())

<details>
<summary>何が起きたか</summary>

`A` は `float64`。欠損があるので整数のままでいられない。
表示が `2.0` `1.0` になり、Parquet に書けば `double` 列になる。

`B` は `Int64`(先頭大文字)。pandas の **nullable 整数型**で、
欠損を `<NA>` として持ちながら整数でいられる。

| | 欠損 | 表示 | 由来 |
| --- | --- | --- | --- |
| `int64` | 持てない | `2` | NumPy |
| `float64` | `NaN` | `2.0` | NumPy |
| `Int64` | `<NA>` | `2` | pandas |

どちらも `sum()` は欠損を無視する。

実務では「最後に欠損を落としてから `int32` にする」ことが多い。
`Int64` は途中の計算で欠損を保ったまま整数を扱いたいときに使う。
`astype("int32")` は欠損があると**落ちる**ので、除外を先に済ませる。

</details>

---
## 5. 日付に変える

### 5-1. format を指定するかどうか

`to_datetime` は書式を推測できるが、推測させると遅く、かつ危ない。

**実行する前に、違いを予想する。**

In [ ]:
# A: 推測させる
s = pd.Series(["2024-01-05", "2024-01-06", ""])
print(pd.to_datetime(s, errors="coerce"))

In [ ]:
# B: 書式を明示する
s = pd.Series(["2024-01-05", "2024-01-06", ""])
print(pd.to_datetime(s, format="%Y-%m-%d", errors="coerce"))

<details>
<summary>何が起きたか</summary>

この例では同じ結果。差が出るのは**曖昧な書式**のとき。

```python
pd.to_datetime(pd.Series(["01/05/2024", "13/05/2024"]), errors="coerce")
```

`01/05/2024` は1月5日か5月1日か決まらない。pandas は列の中身から
推測しようとし、**列によって解釈が変わる**ことがある。
書式が分かっているなら必ず `format=` を書く。速度も一桁違う。

`errors="coerce"` は変換できないものを `NaT` にする。
`NaT` は「日時の欠損」で、`isna()` で判定できる。

</details>

### 5-2. datetime64 と date

Parquet に `date32` で書きたいとき、`datetime64[ns]` のままだと型が合わない。

**実行する前に、違いを予想する。**

In [ ]:
# A: datetime64 のまま
s = pd.to_datetime(pd.Series(["2024-01-05", "2024-01-06"]))
print(s.dtype)
print(s.tolist())

In [ ]:
# B: date にする
s = pd.to_datetime(pd.Series(["2024-01-05", "2024-01-06"]))
d = s.dt.date
print(d.dtype)
print(d.tolist())

<details>
<summary>何が起きたか</summary>

`A` は `datetime64[ns]`、中身は `Timestamp`(時刻を持つ)。
`B` は `object`、中身は `datetime.date`(日付だけ)。

pyarrow のスキーマで `pa.date32()` を要求すると、`A` は
`timestamp[ns]` として扱われて**型チェックに落ちる**。`.dt.date` を通す必要がある。

逆に、日付の差を取ったり月で丸めたりする計算は `A` のほうがやりやすい
(`.dt.to_period("M")` など)。**計算の間は `datetime64`、書き出す直前に `date`**。

</details>

---
## 6. 欠損の比較

### 6-1. NaN は自分自身と等しくない

欠損の判定に `==` を使ってはいけない理由。

**実行する前に、違いを予想する。**

In [ ]:
# A: == で比べる
print(np.nan == np.nan)
s = pd.Series([1.0, np.nan, 3.0])
print(s == np.nan)

In [ ]:
# B: isna で判定する
s = pd.Series([1.0, np.nan, 3.0])
print(s.isna())
print("欠損の数:", s.isna().sum())

<details>
<summary>何が起きたか</summary>

`NaN == NaN` は **False**。IEEE 754 の仕様で、pandas も SQL も同じ考え方。
だから `df[df["col"] == np.nan]` は**必ず0件**になる。エラーは出ない。

欠損は `isna()` / `notna()` で判定する。

欠損の実体は文脈によって3種類ある。

| | どこで出るか |
| --- | --- |
| `np.nan` | float 列 |
| `None` | object 列 |
| `pd.NaT` | datetime 列 |
| `pd.NA` | nullable 型 (`Int64` など) |

`isna()` はどれも `True` にしてくれるので、**判定は常に `isna()`** でよい。

</details>

---
## 練習

上で見たことを使って書く。`assert` が通れば正解。

In [ ]:
# 練習1: orders.csv を「型を推測させずに」読み、
#        NULL / N/A / - / 空文字(前後空白つきも) を欠損にした DataFrame を作る

df = ...   # ここに書く

assert df["amount"].isna().sum() == 1, f"amount の欠損は1件のはず: {df['amount'].isna().sum()}"
assert df["qty"].isna().sum() == 1, f"qty の欠損は1件のはず: {df['qty'].isna().sum()}"
assert df["customer_id"].isna().sum() == 1
assert df["order_date"].isna().sum() == 1
assert df["amount"].dtype == object, "この時点ではまだ文字列のはず"
print("OK")

In [ ]:
# 練習2: 練習1の df の amount を int にする。
#        "2,400" のようなカンマ区切りも通し、欠損は欠損のまま残す。
#        (ヒント: 欠損を含む整数列は Int64)

amount = ...   # ここに書く (Series)

assert str(amount.dtype) == "Int64", f"Int64 のはず: {amount.dtype}"
assert amount.isna().sum() == 1
assert amount.sum() == 23220, f"合計が違う: {amount.sum()}"
print("OK")

In [ ]:
# 練習3: order_date を date にする。欠損は NaT/欠損のまま。

order_date = ...   # ここに書く (Series)

assert order_date.isna().sum() == 1
non_null = order_date.dropna()
assert all(type(v).__name__ == "date" for v in non_null), "datetime.date のはず"
assert str(non_null.iloc[0]) == "2024-01-05"
print("OK")

---
## まとめ

| 引数 / 書き方 | 効果 | いつ使うか |
| --- | --- | --- |
| `dtype=str` | 型を推測させない | 汚れたデータを扱うとき常に |
| `dtype={"id": str}` | 列を名指しで文字列に | IDの先頭ゼロを守る |
| `keep_default_na=False` | 勝手に欠損にしない | 欠損の定義を自分で持つとき |
| `na_values=[...]` | 欠損表現を足す | 読み込み時に閉じたいとき |
| `thousands=","` | 桁区切りを外す | 数値列が単純に汚れているだけのとき |
| `astype(int)` | 失敗したら落ちる | 失敗があってはならないとき |
| `to_numeric(errors="coerce")` | 失敗を NaN にする | 失敗を想定し、**数えるとき** |
| `Int64` | 欠損を持てる整数 | 途中の計算で整数を保ちたいとき |
| `to_datetime(format=...)` | 推測させない | 書式が分かっているとき(常に) |
| `.dt.date` | date32 用に落とす | Parquet に書く直前 |
| `isna()` | 欠損の判定 | `== np.nan` の代わりに常に |

次: `pandas-02-select-and-transform.ipynb`